# Metoda Harmony Search pro úlohu O-CVRPTW

Tento notebook implementuje metodu Harmony Search (HS) adaptovanou na selektivní kapacitní rozvozní úlohu s časovými okny (O-CVRPTW) dle kapitoly 1.5 diplomové práce. Metoda je populační, založená na paměťovém mechanismu (Harmony Memory) a iterativní improvizaci nových řešení.

**Struktura notebooku:**
1. Konfigurace parametrů a načtení datasetu (shodné s MILP notebookem)
2. Evaluátor účelové funkce dle rovnice (8)
3. Konstrukční inicializace (randomized greedy)
4. Operátory pitch adjustment (relocate, swap, 2-opt)
5. Lokální prohledávání a hlavní smyčka Harmony Search
6. Analýza výsledků a souhrnná tabulka

**Požadované knihovny:** pandas, numpy, openpyxl

In [1]:
import math
import random
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any, Optional

import numpy as np
import pandas as pd

# Reprodukovatelnost experimentu (stejné TW jako v MILP)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Cesta k datasetu (nutnost upravit podle aktuálního úložiště souboru)

DATA_PATH = "small_dataset.xlsx"
SHEET_NAME = "small"
DEPOT_CITY_NAME = "City 1"

# Konfigurační třída – MUSÍ odpovídat parametrům MILP

@dataclass(frozen=True)
class Config:
    # Počet vozidel a kapacita (orienteering charakter)
    m: int = 3
    Q: int = 13

    # Jednotková poptávka každého zákazníka
    unit_demand: int = 1

    # Servisní čas u zákazníka
    service_time: float = 1.0

    # Rychlost (speed = 1 → čas = vzdálenost)
    speed: float = 1.0

    # Penalizace z MILP
    PI_LATE: float = 1.5     # penalizace za jednotku zpoždění
    P_SKIP: float = 18.0     # penalizace za neobslouženého zákazníka

    # Parametry generátoru časových oken
    tw_horizon: float = 120.0
    tw_width: float = 20.0
    tw_a_min: float = 0.0
    tw_a_max: float = 80.0

cfg = Config()
print(cfg)


Config(m=3, Q=13, unit_demand=1, service_time=1.0, speed=1.0, PI_LATE=1.5, P_SKIP=18.0, tw_horizon=120.0, tw_width=20.0, tw_a_min=0.0, tw_a_max=80.0)


## 1. Načtení datasetu

Načtení datového souboru small_dataset.xlsx (Příloha 1). Uzel City 1 je umístěn na index 0 jako depot, zbývajících 29 uzlů tvoří množinu zákazníků C = {1, 2, …, 29}. Postup je shodný s MILP notebookem (Příloha 2).

In [2]:
# Načtení vstupního datasetu

df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME).copy()
df_raw.columns = [c.strip() for c in df_raw.columns]

# Sjednocení názvů sloupců
rename_map = {}
if "City" in df_raw.columns: rename_map["City"] = "city"
if "Latitude" in df_raw.columns: rename_map["Latitude"] = "lat"
if "Longitude" in df_raw.columns: rename_map["Longitude"] = "lon"
df_raw = df_raw.rename(columns=rename_map)

# Kontrola povinných sloupců
required = {"city", "lat", "lon"}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f"Chybí sloupce: {missing}")

# Depot musí existovat
if DEPOT_CITY_NAME not in set(df_raw["city"]):
    raise ValueError("Depot nebyl nalezen v datasetu")

# Depot dáme na index 0 (stejně jako v MILP)

depot_row = df_raw[df_raw["city"] == DEPOT_CITY_NAME].iloc[0].to_dict()
others = df_raw[df_raw["city"] != DEPOT_CITY_NAME].copy()

df_nodes = pd.concat([pd.DataFrame([depot_row]), others], ignore_index=True)
df_nodes["node_id"] = range(len(df_nodes))

n = len(df_nodes)
print("Počet uzlů:", n)
display(df_nodes.head())


Počet uzlů: 30


,city,lat,lon,node_id
0,City 1,-0.935834,-1.771533,0
1,City 2,-9.326265,13.902797,1
2,City 3,-8.233092,14.873970,2
3,City 4,3.272331,3.961212,3
4,City 5,1.175829,-0.699877,4


## 2. Vzdálenostní a časová matice

Výpočet symetrické matice euklidovských vzdáleností d_ij a cestovních časů t_ij = d_ij / speed, kde speed = 1,0 dle rovnice (9) v kapitole 1.3.

In [3]:
# Výpočet eukleidovských vzdáleností

def euclid(a_lat, a_lon, b_lat, b_lon) -> float:
    return float(math.sqrt((a_lat - b_lat)**2 + (a_lon - b_lon)**2))

# d_ij – vzdálenost
d = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            d[i, j] = euclid(df_nodes.loc[i, "lat"], df_nodes.loc[i, "lon"],
                             df_nodes.loc[j, "lat"], df_nodes.loc[j, "lon"])

# t_ij – cestovní čas (při speed=1 je rovno vzdálenosti)
t = d / cfg.speed


## 3. Poptávka, servisní časy a časová okna

Jednotková poptávka q_i = 1 pro všechny zákazníky, servisní čas δ = 1,0. Časová okna [e_i, l_i] jsou generována deterministickým generátorem se seedem SEED = 42, shodným s MILP notebookem, s konstantní šířkou 20 časových jednotek a plánovacím horizontem T = 120,0 dle tabulky 2.2 diplomové práce.

In [4]:
# Jednotkové poptávky (depot = 0)

q = np.zeros(n, dtype=int)
q[1:] = cfg.unit_demand

# Servisní časy
service = np.zeros(n)
service[1:] = cfg.service_time

# Generátor časových oken – MUSÍ být shodný s MILP

def generate_time_windows(n: int, seed: int):
    rng = np.random.default_rng(seed)

    ready = np.zeros(n)  # a_i – dolní mez (hard)
    due = np.zeros(n)    # b_i – horní mez (soft)

    ready[0] = 0.0
    due[0] = cfg.tw_horizon

    ready[1:] = rng.uniform(cfg.tw_a_min, cfg.tw_a_max, size=n-1)
    due[1:] = np.minimum(ready[1:] + cfg.tw_width, cfg.tw_horizon)

    return ready, due

ready_time, due_date = generate_time_windows(n, SEED)


## 4. Evaluátor účelové funkce

Implementace účelové funkce Z(R) dle rovnice (8) v kapitole 1.3: součet cestovních nákladů, penalizace za zpoždění (γ · T_i^k) a penalizace za neobsloužení (P_skip). Ekvivalence s fitness funkcí f(R) je prokázána v kapitole 1.5 (rovnice 30). Funkce `eval_route` vyhodnocuje jednotlivou trasu, funkce `eval_solution` agreguje výsledky přes všechny trasy a přidává složku za neobsloužené zákazníky.

In [5]:
def eval_route(route: List[int]) -> Dict[str, float]:
    """
    Vyhodnocení jedné trasy:
    - distance
    - tardiness (od startu obsluhy)
    - kapacita
    """
    travel = 0.0
    tardiness_sum = 0.0
    load = sum(q[i] for i in route if i != 0)

    time_now = max(0.0, ready_time[0])
    prev = route[0]

    for node in route[1:]:
        travel += d[prev, node]
        time_now += t[prev, node]
        arrival = time_now

        if node != 0:
            start = max(arrival, ready_time[node])     # hard lower TW
            tardiness_sum += max(0.0, start - due_date[node])
            time_now = start + service[node]
        else:
            time_now = max(arrival, ready_time[0])

        prev = node

    return {
        "travel": travel,
        "tardiness_sum": tardiness_sum,
        "cap_violation": max(0, load - cfg.Q)
    }


def eval_solution(routes: List[List[int]]) -> Dict[str, Any]:
    """
    Vyhodnocení celého řešení:
    Objective = distance + 1.5*tardiness + 18*unserved
    - tardiness je počítána od startu obsluhy (start = max(arrival, ready_time))
    - unserved je jednorázově penalizováno
    """
    # Množina obsloužených zákazníků (bez depota)
    served = set()
    for r in routes:
        for node in r:
            if node != 0:
                served.add(node)

    # Neobsloužení zákazníci
    customers = set(range(1, n))
    unserved = customers - served

    # Agregace nákladů přes všechny trasy
    travel = 0.0
    tardiness_sum = 0.0
    cap_violation = 0.0

    per_route = []
    for r in routes:
        er = eval_route(r)
        per_route.append(er)
        travel += er["travel"]
        tardiness_sum += er["tardiness_sum"]
        cap_violation += er["cap_violation"]

    # Složky účelové funkce (EXACT parametry)
    tardiness_cost = cfg.PI_LATE * tardiness_sum
    skip_cost = cfg.P_SKIP * len(unserved)

    # Kapacita je v MILP hard constraint – penalizace je pojistka (ideálně vždy 0)
    CAP_PENALTY = 1e6
    cap_cost = CAP_PENALTY * cap_violation

    total = travel + tardiness_cost + skip_cost + cap_cost

    return {
        "total": float(total),
        "travel": float(travel),
        "tardiness_sum": float(tardiness_sum),
        "tardiness_cost": float(tardiness_cost),
        "skip_cost": float(skip_cost),
        "cap_cost": float(cap_cost),
        "cap_violation": float(cap_violation),
        "served_cnt": int(len(served)),
        "unserved_cnt": int(len(unserved)),
        "routes": routes,
        "per_route": per_route,
    }


## 5. Pomocné funkce

Utility funkce pro manipulaci s řešením: inicializace prázdných tras (depot → depot), extrakce zákazníků z trasy, validace řešení (kontrola depotů a duplicit) a hluboká kopie řešení.

In [6]:
def make_empty_routes(m: int) -> List[List[int]]:
    return [[0, 0] for _ in range(m)]

def route_customers(route: List[int]) -> List[int]:
    return [x for x in route if x != 0]

def is_solution_valid(routes: List[List[int]]) -> bool:
    for r in routes:
        if len(r) < 2 or r[0] != 0 or r[-1] != 0:
            return False
    seen = []
    for r in routes:
        seen.extend(route_customers(r))
    return len(seen) == len(set(seen))

def copy_solution(routes: List[List[int]]) -> List[List[int]]:
    return [r.copy() for r in routes]


## 6. Konstrukční inicializace

Randomized greedy konstrukce počátečního řešení pro naplnění paměti harmonie (HM). Zákazníci jsou vkládáni na nejlepší pozici v libovolné trase s respektováním kapacitního omezení Q. Parametr α řídí velikost omezeného seznamu kandidátů (RCL) — vyšší α zvyšuje randomizaci a diverzitu počátečních řešení v HM dle popisu v kapitole 1.5.

In [7]:
def try_insert_customer(routes: List[List[int]], cust: int) -> Optional[Tuple[List[List[int]], float]]:
    base_total = eval_solution(routes)["total"]
    best_new_routes = None
    best_new_total = None

    for k in range(len(routes)):
        r = routes[k]

        if sum(q[x] for x in r if x != 0) + q[cust] > cfg.Q:
            continue

        for pos in range(1, len(r)):
            new_routes = copy_solution(routes)
            new_routes[k] = r[:pos] + [cust] + r[pos:]
            if not is_solution_valid(new_routes):
                continue
            new_total = eval_solution(new_routes)["total"]
            if (best_new_total is None) or (new_total < best_new_total):
                best_new_total = new_total
                best_new_routes = new_routes

    if best_new_routes is None:
        return None
    return best_new_routes, best_new_total

def build_initial_solution(alpha: float = 0.35) -> List[List[int]]:
    routes = make_empty_routes(cfg.m)
    remaining = list(range(1, n))
    random.shuffle(remaining)

    while True:
        current_total = eval_solution(routes)["total"]
        candidates = []

        for cust in remaining:
            res = try_insert_customer(routes, cust)
            if res is None:
                continue
            new_routes, new_total = res
            improvement = current_total - new_total
            if improvement > 1e-9:
                candidates.append((improvement, cust, new_routes))

        if not candidates:
            break

        candidates.sort(key=lambda x: x[0], reverse=True)
        rcl_size = max(1, int(math.ceil(alpha * len(candidates))))
        _, cust, routes = random.choice(candidates[:rcl_size])
        remaining.remove(cust)

    return routes

# smoke test
sol0 = build_initial_solution(alpha=0.4)
ev0 = eval_solution(sol0)
print({k: ev0[k] for k in ["total","travel","tardiness_cost","skip_cost","served_cnt","unserved_cnt","cap_violation"]})
print("valid:", is_solution_valid(sol0))


{'total': 243.57145682822517, 'travel': 61.90583834663925, 'tardiness_cost': 1.665618481585934, 'skip_cost': 180.0, 'served_cnt': 19, 'unserved_cnt': 10, 'cap_violation': 0.0}
valid: True


## 7. Operátory pitch adjustment

Implementace tří operátorů lokálního prohledávání aplikovaných s pravděpodobností PAR dle pseudokódu v kapitole 1.5:
- **Relocate:** přesun náhodného zákazníka z jedné trasy na náhodnou pozici v jiné trase
- **Swap:** výměna dvou náhodně vybraných zákazníků mezi trasami
- **2-opt:** inverze náhodného segmentu uvnitř jedné trasy

Funkce `pitch_adjust` volí operátor rovnoměrně náhodně.

In [8]:
def random_relocate(routes: List[List[int]]) -> List[List[int]]:
    routes = copy_solution(routes)

    src_candidates = [k for k, r in enumerate(routes) if len(route_customers(r)) > 0]
    if not src_candidates:
        return routes
    k_from = random.choice(src_candidates)
    r_from = routes[k_from]

    cust = random.choice(route_customers(r_from))
    routes[k_from] = [x for x in r_from if x != cust]

    k_to = random.randrange(len(routes))
    r_to = routes[k_to]

    if sum(q[x] for x in r_to if x != 0) + q[cust] > cfg.Q:
        routes[k_from] = r_from
        return routes

    pos = random.randrange(1, len(r_to))
    routes[k_to] = r_to[:pos] + [cust] + r_to[pos:]

    if not is_solution_valid(routes):
        return copy_solution(routes)
    return routes

def random_swap(routes: List[List[int]]) -> List[List[int]]:
    routes = copy_solution(routes)

    all_positions = []
    for k, r in enumerate(routes):
        for idx, node in enumerate(r):
            if node != 0:
                all_positions.append((k, idx, node))

    if len(all_positions) < 2:
        return routes

    (k1, i1, _), (k2, i2, _) = random.sample(all_positions, 2)
    routes[k1][i1], routes[k2][i2] = routes[k2][i2], routes[k1][i1]

    if not is_solution_valid(routes):
        return copy_solution(routes)
    return routes

def two_opt_in_route(route: List[int]) -> List[int]:
    if len(route) <= 4:
        return route
    i = random.randrange(1, len(route) - 2)
    j = random.randrange(i + 1, len(route) - 1)
    return route[:i] + list(reversed(route[i:j])) + route[j:]

def random_2opt(routes: List[List[int]]) -> List[List[int]]:
    routes = copy_solution(routes)
    candidates = [k for k, r in enumerate(routes) if len(r) > 4]
    if not candidates:
        return routes
    k = random.choice(candidates)
    routes[k] = two_opt_in_route(routes[k])
    return routes

def pitch_adjust(routes: List[List[int]]) -> List[List[int]]:
    op = random.choice(["relocate", "swap", "2opt"])
    if op == "relocate":
        return random_relocate(routes)
    if op == "swap":
        return random_swap(routes)
    return random_2opt(routes)


## 8. Lokální prohledávání

Intensifikační procedura aplikující opakovaně operátory pitch adjustment (sekce 7) s akceptací zlepšujících tahů (first improvement). Parametr `steps` určuje maximální počet iterací lokálního prohledávání na jedno řešení.

In [9]:
def local_improve_first(routes: List[List[int]], steps: int = 30) -> List[List[int]]:
    best_routes = copy_solution(routes)
    best_val = eval_solution(best_routes)["total"]

    for _ in range(steps):
        neigh = pitch_adjust(best_routes)
        val = eval_solution(neigh)["total"]
        if val + 1e-9 < best_val:
            best_routes = neigh
            best_val = val

    return best_routes


## 9. Hlavní smyčka Harmony Search

Implementace algoritmu dle pseudokódu v kapitole 1.5:
1. **Inicializace HM:** naplnění paměti harmonie HMS řešeními z konstrukční heuristiky (sekce 6)
2. **Improvizace:** s pravděpodobností HMCR je nové řešení odvozeno z HM (memory consideration), jinak je konstruováno náhodně; s pravděpodobností PAR je aplikován pitch adjustment (sekce 7)
3. **Aktualizace HM:** pokud je nové řešení lepší než nejhorší v HM, nahradí ho
4. **Opakování** po NI iterací

Klíčové parametry: HMS (velikost paměti), NI (počet iterací), HMCR (míra využití paměti), PAR (míra lokálního prohledávání).

In [10]:
def hm_initialize(HMS: int, alpha: float = 0.4) -> List[Dict[str, Any]]:
    HM = []
    for _ in range(HMS):
        sol = build_initial_solution(alpha=alpha)
        ev = eval_solution(sol)
        HM.append(ev)
    HM.sort(key=lambda x: x["total"])
    return HM

def hm_worst_index(HM: List[Dict[str, Any]]) -> int:
    return int(np.argmax([x["total"] for x in HM]))

def improvise(HM: List[Dict[str, Any]], HMCR: float, PAR: float, alpha_random: float = 0.6) -> List[List[int]]:
    if random.random() < HMCR:
        base_routes = random.choice(HM)["routes"]
        cand = copy_solution(base_routes)
    else:
        cand = build_initial_solution(alpha=alpha_random)

    if random.random() < PAR:
        cand = pitch_adjust(cand)

    cand = local_improve_first(cand, steps=30)
    return cand

def harmony_search(HMS: int, NI: int, HMCR: float, PAR: float) -> Dict[str, Any]:
    HM = hm_initialize(HMS=HMS, alpha=0.4)
    best = HM[0]
    best_found_iter = 0  # iterace, ve které bylo nalezeno konečné nejlepší řešení
    history = [best["total"]]

    for it in range(1, NI + 1):
        cand_routes = improvise(HM, HMCR=HMCR, PAR=PAR, alpha_random=0.7)
        cand_eval = eval_solution(cand_routes)

        wi = hm_worst_index(HM)
        if cand_eval["total"] + 1e-9 < HM[wi]["total"]:
            HM[wi] = cand_eval
            HM.sort(key=lambda x: x["total"])

        if HM[0]["total"] + 1e-9 < best["total"]:
            best = HM[0]
            best_found_iter = it  # zaznamenání iterace zlepšení

        # průběžný výpis každých 100 iterací
        if it % 100 == 0 or it == 1:
            print(f"Iterace {it}/{NI} | best Z(R) = {best['total']:.2f} | nalezeno v it. {best_found_iter}")

        history.append(best["total"])

    out = dict(best)
    out["history"] = history
    out["HM_final"] = HM
    out["best_found_iter"] = best_found_iter  # předání iterace do výstupního slovníku
    out["NI"] = NI
    return out

## 10. Spuštění experimentu

Spuštění Harmony Search s konfigurací parametrů dle kapitoly 2.4 diplomové práce. Konfigurace se mění mezi jednotlivými běhy přístupem ceteris paribus (tabulky 2.9 a 2.10). Seed SEED = 42 zajišťuje reprodukovatelnost stochastického běhu.

In [11]:
# Spuštění Harmony Search a měření času

start_time = time.time()
best = harmony_search(HMS=20, NI=1000, HMCR=0.90, PAR=0.20) # konfigurace HS
hs_runtime = time.time() - start_time

print("ŘEŠENÍ DOKONČENO")
print(f"Runtime: {hs_runtime:.2f} s")
print(f"Best found in iteration: {best['best_found_iter']} / {best['NI']}")
print(f"Objective: {best['total']:.6f}")

Iterace 1/1000 | best Z(R) = 241.60 | nalezeno v it. 0
Iterace 100/1000 | best Z(R) = 231.87 | nalezeno v it. 86
Iterace 200/1000 | best Z(R) = 219.61 | nalezeno v it. 197
Iterace 300/1000 | best Z(R) = 219.49 | nalezeno v it. 245
Iterace 400/1000 | best Z(R) = 203.05 | nalezeno v it. 376
Iterace 500/1000 | best Z(R) = 155.63 | nalezeno v it. 477
Iterace 600/1000 | best Z(R) = 151.20 | nalezeno v it. 549
Iterace 700/1000 | best Z(R) = 145.36 | nalezeno v it. 680
Iterace 800/1000 | best Z(R) = 143.98 | nalezeno v it. 770
Iterace 900/1000 | best Z(R) = 143.98 | nalezeno v it. 770
Iterace 1000/1000 | best Z(R) = 141.95 | nalezeno v it. 944
ŘEŠENÍ DOKONČENO
Runtime: 25.95 s
Best found in iteration: 944 / 1000
Objective: 141.951905


## 11. Harmonogram tras

Výpočet časového harmonogramu nejlepšího nalezeného řešení: čas příjezdu, čas zahájení obsluhy (s_i^k) a zpoždění (T_i^k) pro každý uzel na trase. Definice je shodná s MILP modelem (podmínky C6–C8).

In [12]:
def compute_route_schedule(route):
    """
    Vypočte arrival, start a tardiness pro každý uzel na trase
    (stejná definice jako v MILP).
    """
    schedule = []
    time_now = max(0.0, ready_time[0])
    prev = route[0]

    for node in route:
        arrival = time_now
        start = max(arrival, ready_time[node])
        tau = max(0.0, start - due_date[node])
        depart = start + (service[node] if node != 0 else 0.0)

        schedule.append((node, start, tau))
        time_now = depart
        prev = node

    return schedule


## 12. Rozklad účelové funkce

Ověření hodnoty účelové funkce Z(R) rozkladem na jednotlivé složky dle rovnice (8): cestovní náklady dist(R), penalizace za zpoždění γ · tard(R) a penalizace za neobsloužení P_skip · |C \ served(R)|.

In [13]:
rec = eval_solution(best["routes"])

print("Objective breakdown (EXACT):")
print("distance     =", rec["travel"])
print("1.5*tardiness=", cfg.PI_LATE * rec["tardiness_sum"])
print("18*unserved  =", rec["skip_cost"])


Objective breakdown (EXACT):
distance     = 84.78232988533318
1.5*tardiness= 3.169575479989671
18*unserved  = 54.0


## 13. Rekalkulace účelové funkce

Nezávislá rekalkulace účelové funkce ve formátu konzistentním s MILP reportem pro zajištění srovnatelnosti výsledků v kapitole 2.5.

In [14]:
def hs_recompute_objective(routes: List[List[int]]) -> Dict[str, Any]:
    """
    Rekalkulace účelové funkce HS ve stejném tvaru,
    jaký očekává MILP-like report.
    """
    # Obsloužení / neobsloužení
    served = set()
    for r in routes:
        for node in r:
            if node != 0:
                served.add(node)

    customers = set(range(1, n))
    unserved = customers - served

    # Agregace nákladů
    travel_cost = 0.0
    tardiness_sum = 0.0
    cap_violation = 0.0

    for r in routes:
        er = eval_route(r)
        travel_cost += er["travel"]
        tardiness_sum += er["tardiness_sum"]
        cap_violation += er["cap_violation"]

    # EXACT parametry (stejné jako MILP)
    tardiness_cost = cfg.PI_LATE * tardiness_sum
    skip_cost = cfg.P_SKIP * len(unserved)

    # Rekonstruovaná účelová funkce (bez cap penalty)
    objective_recomputed = travel_cost + tardiness_cost + skip_cost

    return {
        "objective_recomputed": float(objective_recomputed),

        # Detailní rozpad
        "travel_cost": float(travel_cost),
        "tardiness_sum": float(tardiness_sum),
        "tardiness_cost": float(tardiness_cost),
        "skip_cost": float(skip_cost),

        # Statistiky
        "served_cnt": int(len(served)),
        "unserved_cnt": int(len(unserved)),
        "cap_violation": float(cap_violation),
    }


rec = hs_recompute_objective(best["routes"])

print("Objective breakdown (EXACT):")
print(f"total         = {rec['objective_recomputed']:.6f}")
print(f"distance      = {rec['travel_cost']:.6f}")
print(f"1.5*tardiness = {rec['tardiness_cost']:.6f}  (tardiness_sum = {rec['tardiness_sum']:.6f})")
print(f"18*unserved   = {rec['skip_cost']:.6f}  (unserved = {rec['unserved_cnt']})")
print(f"cap_violation = {rec['cap_violation']:.1f}")


Objective breakdown (EXACT):
total         = 141.951905
distance      = 84.782330
1.5*tardiness = 3.169575  (tardiness_sum = 2.113050)
18*unserved   = 54.000000  (unserved = 3)
cap_violation = 0.0


## 14. Souhrnná tabulka výsledků

Agregace výsledků do jednotného formátu pro srovnání s referenčním MILP modelem a ostatními heuristickými metodami (kapitola 2.5). Tabulka obsahuje hodnotu Z(R), rozklad na složky, počet obsloužených zákazníků a dobu výpočtu. Součástí shrnutí je také informace o pořadovém čísle konfigurace, ve kterém metoda našla optimální řešení.

In [15]:
# VÝSLEDNÁ TABULKA (HS)

result_row_hs = {
    "method": "HS",
    # Objective pro benchmarking bereme z recomputed (stejně jako MILP kontrola)
    "objective": rec["objective_recomputed"],

    # Rozpad složek cíle
    "distance": rec["travel_cost"],
    "tardiness_sum": rec["tardiness_sum"],
    "tardiness_cost": rec["tardiness_cost"],
    "unserved": rec["unserved_cnt"],
    "skip_cost": rec["skip_cost"],

    # Další KPI
    "served": rec["served_cnt"],
    "cap_violation": rec["cap_violation"],

    # Runtime
    "runtime_sec": hs_runtime,

    # Iterace nalezení nejlepšího řešení
    "best_found_iter": best["best_found_iter"],
    "NI": best["NI"],
}

df_result_hs = pd.DataFrame([result_row_hs])
display(df_result_hs)

# Výpis tras vozidel (stejný formát jako ACO a BS)
served_nodes = set()
for route in best["routes"]:
    for node in route:
        if node != 0:
            served_nodes.add(node)

total_customers = n - 1  # bez depota
print(f"\nServed customers: {len(served_nodes)} / {total_customers}")
print(f"Best found in iteration: {best['best_found_iter']} / {best['NI']}")
print("Routes:")
for k, route in enumerate(best["routes"], 1):
    print(f"Vehicle {k}: {route}")

,method,objective,distance,tardiness_sum,tardiness_cost,unserved,skip_cost,served,cap_violation,runtime_sec,best_found_iter,NI
0,HS,141.951905,84.78233,2.11305,3.169575,3,54.0,26,0.0,25.951989,944,1000



Served customers: 26 / 29
Best found in iteration: 944 / 1000
Routes:
Vehicle 1: [0, 9, 29, 22, 15, 16, 21, 17, 25, 7, 19, 0]
Vehicle 2: [0, 26, 5, 18, 2, 11, 10, 27, 24, 8, 12, 3, 6, 0]
Vehicle 3: [0, 28, 20, 4, 13, 0]
